In [2]:
import pandas as pd
import glob
import os
import numpy as np

# --- CONFIGURACIÓN ---
# Cambia esto si tu notebook no está justo al lado de la carpeta "resultados"
ruta_base = 'resultados/' 
clases = ['car', 'chair', 'person', 'pizza', 'TV', 'umbrella']

# --- LISTAS PARA ALMACENAR DATOS ---
cfg_list = []
rescale_list = []
steps_list = []
convergence_gens = []
execution_times_mins = []

for clase in clases:
    clase_path = os.path.join(ruta_base, clase)
    if not os.path.exists(clase_path):
        continue
        
    print(f"Procesando clase: {clase}...")

    # 1. PARÁMETROS ÓPTIMOS (Desde los Paretos)
    pareto_files = glob.glob(os.path.join(clase_path, 'paretos', '*.csv'))
    for pf in pareto_files:
        df_pareto = pd.read_csv(pf)
        if not df_pareto.empty:
            # Asumimos como "Mejor Individuo" el que tiene mayor Fitness YOLO
            best_idx = df_pareto['fitness_yolo'].idxmax()
            best_ind = df_pareto.loc[best_idx]
            
            cfg_list.append(best_ind['cfg'])
            rescale_list.append(best_ind['guidance_rescale'])
            steps_list.append(best_ind['f2']) # f2 es Num Inference Steps
            
    # 2. CONVERGENCIA (Desde history.csv)
    history_file = os.path.join(clase_path, 'history.csv')
    if os.path.exists(history_file):
        df_hist = pd.read_csv(history_file)
        # Iteramos por cada run_id (de 1 a 10)
        for run_id in df_hist['run_id'].unique():
            df_run = df_hist[df_hist['run_id'] == run_id]
            max_hv = df_run['hypervolume_feas'].max()
            
            # Definimos convergencia: momento en el que se alcanza el 98% del HV máximo
            umbral = max_hv * 0.98
            df_supera = df_run[df_run['hypervolume_feas'] >= umbral]
            if not df_supera.empty:
                gen_conv = df_supera['gen'].min()
                convergence_gens.append(gen_conv)

    # 3. TIEMPO DE EJECUCIÓN (Desde runs_summary.csv)
    # Como no hay una columna de duración exacta, calculamos la diferencia 
    # entre los timestamps de finalización de cada ejecución.
    summary_file = os.path.join(clase_path, 'runs_summary.csv')
    if os.path.exists(summary_file):
        df_sum = pd.read_csv(summary_file)
        # Convertimos la columna a formato fecha/hora y ordenamos
        df_sum['timestamp'] = pd.to_datetime(df_sum['timestamp'])
        df_sum = df_sum.sort_values('timestamp')
        
        # Calculamos la diferencia de tiempo entre runs consecutivos en minutos
        diffs = df_sum['timestamp'].diff().dt.total_seconds() / 60.0
        # Filtramos posibles saltos de días si paraste el experimento y lo retomaste (ej. diff > 300 min)
        diffs_validas = diffs[(diffs > 0) & (diffs < 300)].dropna().tolist()
        execution_times_mins.extend(diffs_validas)


# --- CÁLCULO FINAL Y PRESENTACIÓN ---
print("\n" + "="*50)
print("🎯 MÉTRICAS FINALES PARA COMPARAR CON STABLEYOLO")
print("="*50)

print(f"➤ Guidance Scale (CFG):      {np.mean(cfg_list):.2f} ± {np.std(cfg_list):.2f} (Rango de sus resultados: [0, 20])")
print(f"➤ Guidance Rescale:          {np.mean(rescale_list):.2f} ± {np.std(rescale_list):.2f}")
print(f"➤ Inference Steps (f2):      {np.mean(steps_list):.2f} ± {np.std(steps_list):.2f} (Ellos reportan 45.3 ± 30.29)")
print(f"➤ Iteraciones convergencia:  {np.mean(convergence_gens):.2f} ± {np.std(convergence_gens):.2f} (Ellos reportan 45.05 ± 10.01)")

if execution_times_mins:
    print(f"➤ Tiempo aprox. por run:     {np.mean(execution_times_mins):.2f} minutos (Ellos reportan ~2h por prompt)")
else:
    print("➤ Tiempo aprox. por run:     No se pudo calcular la diferencia de timestamps.")

Procesando clase: car...
Procesando clase: chair...
Procesando clase: person...
Procesando clase: pizza...
Procesando clase: TV...
Procesando clase: umbrella...

🎯 MÉTRICAS FINALES PARA COMPARAR CON STABLEYOLO
➤ Guidance Scale (CFG):      10.19 ± 4.85 (Rango de sus resultados: [0, 20])
➤ Guidance Rescale:          0.47 ± 0.32
➤ Inference Steps (f2):      28.53 ± 21.86 (Ellos reportan 45.3 ± 30.29)
➤ Iteraciones convergencia:  20.97 ± 8.78 (Ellos reportan 45.05 ± 10.01)
➤ Tiempo aprox. por run:     35.09 minutos (Ellos reportan ~2h por prompt)
